# DAC Multi-Tile Sync Example

Generate matching DAC0 and DAC2 waveforms, synchronize the active DAC tiles with RFDC MTS, then enable both outputs. Without MTS, the two DAC tile datapaths can start with different latency, so the observed output phases are not guaranteed to match after each initialization.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from firmware import OverlayController
from firmware import signals

ol = OverlayController()
info = ol.info()
info

## Active RFDC Tiles

This bitstream only enables DAC tiles 0 and 2, with block 0 active on each tile. The summary below only queries those active resources.

In [ ]:
def print_rfdc_summary():
    rfdc = ol.xrfdc
    print("IPStatus:", rfdc.IPStatus)

    for tile_id in [0, 2]:
        tile = rfdc.dac_tiles[tile_id]
        print(f"DAC tile {tile_id}: PLLLockStatus={tile.PLLLockStatus}, FIFOStatus={tile.FIFOStatus}")

        block_id = 0
        st = tile.blocks[block_id].BlockStatus
        print(
            f"  block {block_id}: SamplingFreq={st.get('SamplingFreq')}, "
            f"DigitalPathEnabled={st.get('DigitalPathEnabled')}, "
            f"DataPathClocksStatus={st.get('DataPathClocksStatus')}"
        )


print_rfdc_summary()

In [ ]:
DAC0_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
DAC2_SR = float(info["rfdc"]["dac2_sampling_rate_gsps"]) * 1e9
DAC0_LEN = int(info["dac0"]["bram_int16_samples"])
DAC_PEAK = int(0.8 * np.iinfo(np.int16).max)
SAMPLES_PER_VECTOR = 512 // 16

DAC0_SR, DAC2_SR, DAC0_LEN, DAC_PEAK

In [ ]:
def to_int16(waveform, peak=DAC_PEAK):
    data = np.asarray(waveform, dtype=float)
    return np.round(np.clip(data, -float(peak), float(peak))).astype(np.int16)


def samples_for_duration(duration_s, sample_rate_hz, player):
    samples = int(round(float(duration_s) * float(sample_rate_hz)))
    if samples <= 0:
        raise ValueError("duration must produce at least one sample")
    if samples > player.capacity:
        raise ValueError(
            f"duration requires {samples} samples, but {player.name} capacity is {player.capacity}"
        )
    return samples


def vectors_for_samples(num_samples):
    return int(np.ceil(int(num_samples) / SAMPLES_PER_VECTOR))


def read_dac0_length_raw():
    return int(ol.gpio_control.axi_gpio_dac.mmio.read(0x8)) & ol.dac0.length_mask


def plot_waveform(waveform, sample_rate, samples=2048, title="Waveform"):
    view = np.asarray(waveform)[:samples]
    time_ns = np.arange(view.size) / sample_rate * 1e9

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time_ns, view)
    ax.set_title(title)
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("DAC code")
    ax.grid(True, alpha=0.3)
    return ax

## Generate Matched Waveforms

Both DACs are loaded with the same sine wave and active loop length. DAC0 still uses the full zero-padded BRAM write pattern from the simple example, then its active hardware loop length is set separately.

In [ ]:
requested_total_seconds = 1.0e-6
tone_hz = 100e6

if not np.isclose(DAC0_SR, DAC2_SR):
    raise RuntimeError(f"DAC sample rates differ: DAC0={DAC0_SR}, DAC2={DAC2_SR}")

active_samples = samples_for_duration(requested_total_seconds, DAC0_SR, ol.dac0)
matched_waveform = to_int16(
    signals.sine(freq_hz=tone_hz, sample_rate=DAC0_SR, num_samples=active_samples, amplitude=DAC_PEAK)
)

dac0_waveform = np.zeros(DAC0_LEN, dtype=np.int16)
dac0_waveform[:active_samples] = matched_waveform
dac2_waveform = matched_waveform.copy()

print(f"Tone: {tone_hz / 1e6:.3f} MHz")
print(f"Active loop length: {active_samples} samples")
print(f"Active vectors: {vectors_for_samples(active_samples)} vectors")
print(f"DAC0 BRAM write length: {len(dac0_waveform)} samples")
print(f"DAC2 BRAM write length: {len(dac2_waveform)} samples")

matched_waveform.dtype, matched_waveform.shape


In [ ]:
plot_waveform(matched_waveform, DAC0_SR, title="Matched DAC0/DAC2 active waveform");

## Program DAC Players

Disable both outputs before loading waveforms. Both hardware loop lengths are set to the same active sample count.

In [ ]:
ol.dac0.disable()
ol.dac2.disable()
time.sleep(0.01)

ol.dac0.load_waveform(dac0_waveform)
ol.dac0.set_waveform_length(active_samples)
length_readback = read_dac0_length_raw()
if length_readback != active_samples:
    raise RuntimeError(
        f"DAC0 length mismatch: wrote {active_samples:#x}, read {length_readback:#x}"
    )

ol.dac2.load_waveform(dac2_waveform)
ol.dac2.set_waveform_length(active_samples)

ol.info()

## Synchronize DAC Tiles

Run RFDC multi-tile synchronization before enabling output. This aligns DAC tile 0 and DAC tile 2 datapath latency against the configured reference tile.

In [ ]:
mts_status = ol.sync_dac_tiles()
print("MTS status:", mts_status)
ol.dac_mts_info()

## Enable Outputs

Only run this cell when the RF chain and instruments are ready. Measure DAC0 and DAC2 on a phase-coherent instrument to verify the aligned output phase.

In [ ]:
ol.dac0.enable()
ol.dac2.enable()

ol.dac0.is_enabled(), ol.dac2.is_enabled()

## Disable Outputs

Run this before changing cabling or loading a different waveform.

In [ ]:
ol.dac0.disable()
ol.dac2.disable()

ol.info()